# Problem 2 — Transformer from tensor operations

The implementation uses only Linear, LayerNorm, Embedding, tensor operations, and a handwritten attention function. No `nn.MultiheadAttention` or `nn.Transformer*` modules are used. The fast path runs numerical tests and reconciles block parameter counts. Set `RUN_FULL_EXPERIMENTS = True` on a GPU to run HAR position ablations, character-LM training and generation, attention plots, head/mask ablations, and context scaling.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == 'problem2':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from problem2.solution import *
RUN_FULL_EXPERIMENTS = False  # Switch to True on a GPU for the full marked experiments.

## 2.1 Scaled dot-product attention tests

The mask is an allowed-position boolean tensor; causal attention masks future keys. The hand example uses zero query/key vectors, so allowed weights are uniform and the output can be checked by hand.

In [2]:
attention_test = test_attention()
print(attention_test)
print('Hand-check: outputs are [1,0], [1/2,1/2], [2/3,2/3].')

{'max_sdpa_error': 2.384185791015625e-07}
Hand-check: outputs are [1,0], [1/2,1/2], [2/3,2/3].


## 2.2 Multi-head attention, encoder block and parameter count

In [3]:
small_block_counts = test_encoder_block()
standard_block_counts = encoder_block_parameter_count(128, 512)
assert standard_block_counts['formula_weights'] == 196608
assert standard_block_counts['module_total'] == standard_block_counts['formula_total']
print('Shape test:', small_block_counts)
print('d=128, d_ff=512:', standard_block_counts)
print('Weights = 4*d^2 + 2*d*d_ff; biases and two LayerNorm affine pairs are counted separately.')

Shape test: {'formula_weights': 12288, 'formula_biases': 288, 'formula_layer_norm': 128, 'formula_total': 12704, 'module_total': 12704}
d=128, d_ff=512: {'formula_weights': 196608, 'formula_biases': 1152, 'formula_layer_norm': 512, 'formula_total': 198272, 'module_total': 198272}
Weights = 4*d^2 + 2*d*d_ff; biases and two LayerNorm affine pairs are counted separately.


## Full experiments (GPU recommended)

HAR uses the same subject-wise validation split and training-only channel statistics as Problem 1. The language model uses the first 90% of Tiny Shakespeare for training and last 10% for validation, with context 128, d=128, 4 heads, and 4 layers.

In [4]:
if RUN_FULL_EXPERIMENTS:
    har_transformer_rows = []
    for position_mode in ('none', 'sinusoidal', 'learned'):
        for experiment_seed in (0, 1, 2):
            row, model, history = train_har_transformer(position_mode, seed=experiment_seed, epochs=8)
            har_transformer_rows.append(row)
    display(pd.DataFrame(har_transformer_rows))

    lm, lm_history = train_language_model(context=128, n_heads=4, n_layers=4,
                                          causal=True, max_steps=1000, seed=0)
    plot_lm_curve(lm_history)
    lm_row = record_language_result(lm_history, lm, notes='context=128; heads=4; layers=4; causal')
    print(lm_row, 'bits/char = validation nats / ln(2)')
    _, stoi, itos, _, _ = load_tiny_shakespeare()
    prompt = torch.tensor([[stoi.get('\n', 0)]])
    generated = {}
    for temperature in (0.2, 0.8, 1.5):
        generated[temperature] = generate(lm, prompt, itos, length=499,
                                           temperature=temperature, top_k=20)[-500:]
    with open('problem2/generated_samples.txt', 'w', encoding='utf-8') as output_file:
        for temperature, sample in generated.items():
            assert len(sample) == 500
            output_file.write(f'Temperature {temperature}:\n{sample}\n\n')
    plot_attention_heatmaps(lm, 'To be, or not to be.', stoi)

    heads_ablation = run_heads_ablation((1, 4, 8), max_steps=500)
    display(heads_ablation)
    no_mask_model, no_mask_history = train_language_model(
        context=128, n_heads=4, n_layers=4, causal=False, max_steps=1000, seed=0)
    plot_lm_curve(no_mask_history, name='no_mask')
    no_mask_row = record_language_result(
        no_mask_history, no_mask_model, notes='context=128; heads=4; layers=4; NO causal mask (future leakage)')
    print('No-mask result:', no_mask_row)
    context_results = run_context_benchmark((64, 128, 256, 512), batch_size=4)
    display(context_results)
    print('Figures saved to problem2/figures; run rows saved to results.csv.')
else:
    print('Full experiments skipped. Set RUN_FULL_EXPERIMENTS=True on a GPU to execute them.')

Full experiments skipped. Set RUN_FULL_EXPERIMENTS=True on a GPU to execute them.


## Interpretation checklist

Discuss the two head heatmaps as query-to-key relationships, not as causal explanations. Report validation loss in nats and bits per character (nats divided by ln 2). The unmasked model is deliberately invalid for next-character generation because it sees future characters during training; its lower training loss is leakage, not a better causal model. Context attention-score storage grows quadratically with sequence length, while the projections and feed-forward operations also depend on d.